# PS S6E6 048: One-vs-Rest XGB seed/FE/fold variant

- Experiment: `exp_20260614_048_ovr_xgb_seed2027_fe_foldvariant`
- Base experiment: `exp_20260608_030_ovr_xgb_as_is`
- Purpose: return to the binary one-vs-rest XGB route after the failed 047 multiclass softprob test.
- Main changes from 030: seed/fold randomization, modest additional FE, fold-specific TargetEncoder seeds, and mild target-specific `scale_pos_weight`.
- No external stacking, no LogisticRegressionCV second-stage blend, no bias search for final output.
- Output filenames include exp_id and experiment number 048.


In [1]:
# ============================================================
# 0. Imports / config
# ============================================================
import os
import gc
import json
import random
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder, KBinsDiscretizer
from sklearn.metrics import balanced_accuracy_score, recall_score, roc_auc_score, confusion_matrix
from xgboost import XGBClassifier

try:
    import yaml
except Exception:
    yaml = None

COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260614_048_ovr_xgb_seed2027_fe_foldvariant"
EXP_NO = "048"
SEED = 2027
SEED_LIST = [2027, 3407, 7117]
FOLDS = 5
FOLD_SEED_MULTIPLIER = 17
SPW_POWER = 0.25

ID = "id"
TARGET = "class"
CLASS_NAMES = ["GALAXY", "QSO", "STAR"]
target_mapping = {"GALAXY": 0, "QSO": 1, "STAR": 2}
reverse_mapping = {0: "GALAXY", 1: "QSO", 2: "STAR"}

NUMS = ["alpha", "delta", "u", "g", "r", "i", "z", "redshift"]
BANDS = ["u", "g", "r", "i", "z"]
CATS = ["spectral_type", "galaxy_population"]

WORKDIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUTDIR = WORKDIR / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

OOF_NPY = OUTDIR / f"oof_{EXP_ID}_proba.npy"
PRED_NPY = OUTDIR / f"pred_{EXP_ID}_proba.npy"
OOF_CSV = OUTDIR / f"oof_{EXP_ID}.csv"
PRED_CSV = OUTDIR / f"pred_{EXP_ID}.csv"
SUB_CSV = OUTDIR / f"submission_{EXP_ID}.csv"
CV_RESULT_JSON = OUTDIR / f"cv_result_{EXP_ID}.json"
FOLD_SCORES_CSV = OUTDIR / f"fold_scores_{EXP_ID}.csv"
FEATURE_INFO_JSON = OUTDIR / f"feature_info_{EXP_ID}.json"
FEATURE_COLUMNS_CSV = OUTDIR / f"feature_columns_{EXP_ID}.csv"
MODEL_LOG_CSV = OUTDIR / f"binary_model_log_{EXP_ID}.csv"
CLASS_DIST_CSV = OUTDIR / f"class_distribution_{EXP_ID}.csv"

def seed_everything(seed: int = 42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

seed_everything(SEED)

print("EXP_ID:", EXP_ID)
print("OUTDIR:", OUTDIR)
print("SEED_LIST:", SEED_LIST)


EXP_ID: exp_20260614_048_ovr_xgb_seed2027_fe_foldvariant
OUTDIR: /kaggle/working/exp_20260614_048_ovr_xgb_seed2027_fe_foldvariant
SEED_LIST: [2027, 3407, 7117]


In [2]:
# ============================================================
# 1. Load competition data
# ============================================================
candidate_dirs = [
    Path("/kaggle/input/competitions/playground-series-s6e6"),
    Path("/kaggle/input/playground-series-s6e6"),
    Path("/kaggle/input/playground-series-season-6-episode-6"),
]

DATA_DIR = None
for d in candidate_dirs:
    if (d / "train.csv").exists() and (d / "test.csv").exists() and (d / "sample_submission.csv").exists():
        DATA_DIR = d
        break

if DATA_DIR is None:
    raise FileNotFoundError("Could not find train.csv/test.csv/sample_submission.csv under known Kaggle input paths.")

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

train = pd.read_csv(TRAIN_PATH, index_col=ID)
test = pd.read_csv(TEST_PATH, index_col=ID)
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)

train[TARGET] = train[TARGET].map(target_mapping).astype("int32")

print("DATA_DIR:", DATA_DIR)
print("train:", train.shape)
print("test:", test.shape)
print("sample_submission:", sample_submission.shape)
print(train[TARGET].value_counts().sort_index())


DATA_DIR: /kaggle/input/competitions/playground-series-s6e6
train: (577347, 11)
test: (247435, 10)
sample_submission: (247435, 2)
class
0    377480
1    117143
2     82724
Name: count, dtype: int64


In [3]:
# ============================================================
# 2. Feature engineering: conservative OVR XGB variant
# ============================================================
cat_cols = train.select_dtypes(include=["object"]).columns.tolist()
num_cols = train.select_dtypes(exclude=["object"]).columns.tolist()
cat_cols = [col for col in cat_cols if col not in [ID, TARGET]]
num_cols = [col for col in num_cols if col not in [ID, TARGET]]

print("init len(cat_cols):", len(cat_cols))
print("init len(num_cols):", len(num_cols))

category_map = {}

# Keep 030 core color pairs and add a small number of adjacent color indices.
color_pairs = [
    ("u", "g"),
    ("u", "r"),
    ("g", "r"),
    ("r", "i"),
    ("i", "z"),
    ("g", "i"),
    ("r", "z"),
]

def safe_div(a, b, eps=1e-6):
    return a / (b + eps)

def feature_engineering(df: pd.DataFrame, fit: bool = False) -> pd.DataFrame:
    df = df.copy()

    # Core 030 arithmetic interactions.
    df["_g_/_redshift"] = safe_div(df["g"], df["redshift"]).astype("float32")
    df["_i_/_redshift"] = safe_div(df["i"], df["redshift"]).astype("float32")

    # Conservative extra color/magnitude/redshift features.
    for a, b in color_pairs:
        df[f"_{a}-{b}"] = (df[a] - df[b]).astype("float32")

    band_values = df[BANDS].astype("float32")
    df["_mag_mean"] = band_values.mean(axis=1).astype("float32")
    df["_mag_std"] = band_values.std(axis=1).astype("float32")
    df["_mag_range"] = (band_values.max(axis=1) - band_values.min(axis=1)).astype("float32")
    df["_mag_slope_uz"] = ((df["z"] - df["u"]) / 4.0).astype("float32")
    df["_redshift_abs"] = np.abs(df["redshift"]).astype("float32")
    df["_log1p_redshift"] = np.log1p(np.clip(df["redshift"], -0.999999, None)).astype("float32")
    df["_z_/_g"] = safe_div(df["z"], df["g"]).astype("float32")

    # Categorize string categoricals.
    for col in cat_cols:
        if fit:
            codes, uniques = df[col].factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = df[col].map(code_map).fillna(-1).astype("int32")
        df[col] = codes
        df[col] = df[col].astype("category")

    # Categorize floored numericals.
    for col in num_cols:
        cat_name = f"{col}_cat_"
        floored = np.floor(df[col])
        if fit:
            codes, uniques = floored.factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = floored.map(code_map).fillna(-1).astype("int32")
        df[cat_name] = codes
        df[cat_name] = df[cat_name].astype("category")

    # Keep 030 delta bins and add one low-cost redshift bin.
    bin_config = {
        "delta": [100, 500],
        "redshift": [64],
    }
    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            bin_name = f"{col}_{n_bins}_quantile_bin_"
            if fit:
                kb = KBinsDiscretizer(
                    n_bins=n_bins,
                    encode="ordinal",
                    strategy="quantile",
                    subsample=None,
                )
                binned = kb.fit_transform(df[[col]]).ravel().astype("int32")
                category_map[bin_name] = kb
            else:
                kb = category_map[bin_name]
                binned = kb.transform(df[[col]]).ravel().astype("int32")
            df[bin_name] = binned
            df[bin_name] = df[bin_name].astype("category")

    return df

comb = feature_engineering(pd.concat([train.drop(columns=TARGET), test], ignore_index=True), fit=True)
y_series = train[TARGET].copy().reset_index(drop=True)
test = comb.iloc[len(y_series):].reset_index(drop=True)
train_fe = comb.iloc[:len(y_series)].reset_index(drop=True)
train_fe[TARGET] = y_series.copy()
del comb
gc.collect()

new_cat_cols = [col for col in train_fe.columns if col.endswith("_")]
new_num_cols = [col for col in train_fe.columns if col.startswith("_")]

DROP = [c for c in test.columns if test[c].nunique() == 1]
print("DROP:", DROP)
if DROP:
    train_fe.drop(DROP, axis=1, inplace=True)
    test.drop(DROP, axis=1, inplace=True)
    new_cat_cols = [col for col in new_cat_cols if col not in DROP]

CATEGORY = CATS + [c for c in test.columns if "digit" in c]
for c in CATEGORY:
    if c not in train_fe.columns:
        continue
    freq = train_fe[c].value_counts()
    mapping = {val: idx for idx, (val, count) in enumerate(freq[freq >= 5].items())}
    mapping_default = len(mapping)
    train_fe[c] = train_fe[c].map(lambda x: mapping.get(x, mapping_default)).astype("int32")
    test[c] = test[c].map(lambda x: mapping.get(x, mapping_default)).astype("int32")

NUMS_EFFECTIVE = [c for c in NUMS if c in train_fe.columns]
CATS_EFFECTIVE = [c for c in CATS if c in train_fe.columns]

print("train_fe:", train_fe.shape)
print("test_fe:", test.shape)
print("new_cat_cols:", len(new_cat_cols))
print("new_num_cols:", len(new_num_cols))


init len(cat_cols): 2
init len(num_cols): 8
DROP: []
train_fe: (577347, 38)
test_fe: (247435, 37)
new_cat_cols: 11
new_num_cols: 16


In [4]:
# ============================================================
# 3. Combo features for fold-safe TargetEncoder
# ============================================================
TE_columns = []

important_combos = [
    ("alpha_cat_", "delta_cat_"),
    ("u_cat_", "z_cat_"),
    ("redshift_64_quantile_bin_", "g_cat_"),
]
important_combos = sorted(important_combos)

for cols in important_combos:
    if not all(c in train_fe.columns and c in test.columns for c in cols):
        print("skip combo due to missing columns:", cols)
        continue

    name = "-".join(cols)

    train_fe[name] = train_fe[cols[0]].astype(str)
    for col in cols[1:]:
        train_fe[name] = train_fe[name] + "_" + train_fe[col].astype(str)

    test[name] = test[cols[0]].astype(str)
    for col in cols[1:]:
        test[name] = test[name] + "_" + test[col].astype(str)

    combined = pd.concat([train_fe[name], test[name]], ignore_index=True)
    combined_codes, _ = combined.factorize()

    if pd.Series(combined_codes).nunique() > len(combined_codes) // 2:
        train_fe = train_fe.drop(name, axis=1)
        test = test.drop(name, axis=1)
        print("drop high-card combo:", name)
        continue

    train_fe[name] = combined_codes[:len(train_fe)].astype("int32")
    test[name] = combined_codes[len(train_fe):len(train_fe) + len(test)].astype("int32")
    TE_columns.append(name)

print("TE_columns:", TE_columns)


TE_columns: ['alpha_cat_-delta_cat_', 'redshift_64_quantile_bin_-g_cat_', 'u_cat_-z_cat_']


In [5]:
# ============================================================
# 4. Prepare matrices
# ============================================================
train_full = train_fe.copy()
test_df = test.copy().reset_index(drop=True)

train_full["id"] = train_full.index
train_full[TARGET] = train_full[TARGET].map(reverse_mapping)
test_df["id"] = sample_submission["id"].values

y = train_full[TARGET].map(target_mapping).values.astype("int32")
target_ovr = pd.DataFrame({
    "GALAXY": (y == 0).astype("int32"),
    "QSO": (y == 1).astype("int32"),
    "STAR": (y == 2).astype("int32"),
})

cat_cols_final = [c for c in (CATS + new_cat_cols) if c in train_full.columns and c != TARGET]
feature_cols = [c for c in train_full.columns if c not in ["id", TARGET] and train_full[c].dtype != "object"]
feature_cols = cat_cols_final + [c for c in feature_cols if c not in cat_cols_final]

features = train_full[feature_cols].copy()
test_features = test_df[feature_cols].copy()
test_ids = test_df["id"].values

# Encode category dtypes as int codes consistently.
for c in cat_cols_final:
    if str(features[c].dtype) == "category":
        combined_cats = pd.concat([features[c], test_features[c]]).astype(str).unique()
        features[c] = pd.Categorical(features[c].astype(str), categories=combined_cats).codes.astype("int32")
        test_features[c] = pd.Categorical(test_features[c].astype(str), categories=combined_cats).codes.astype("int32")
    else:
        features[c] = features[c].astype("int32")
        test_features[c] = test_features[c].astype("int32")

feature_info = {
    "n_features": int(len(feature_cols)),
    "n_cat_cols_final": int(len(cat_cols_final)),
    "n_te_columns": int(len(TE_columns)),
    "feature_cols": feature_cols,
    "cat_cols_final": cat_cols_final,
    "te_columns": TE_columns,
    "source": "ps6e6-one-vs-rest-xgb.ipynb XGB body only",
}

pd.DataFrame({"feature": feature_cols}).to_csv(FEATURE_COLUMNS_CSV, index=False)
with open(FEATURE_INFO_JSON, "w", encoding="utf-8") as f:
    json.dump(feature_info, f, indent=2, ensure_ascii=False, default=str)

print("features:", features.shape)
print("test_features:", test_features.shape)
print("Total features:", len(feature_cols))
print("TE columns:", TE_columns)


features: (577347, 40)
test_features: (247435, 40)
Total features: 40
TE columns: ['alpha_cat_-delta_cat_', 'redshift_64_quantile_bin_-g_cat_', 'u_cat_-z_cat_']


In [6]:
# ============================================================
# 5. XGB one-vs-rest training: conservative seed/FE/fold variant
# ============================================================
def xgb_params_for(seed: int, target_id: int | None = None, y_train: pd.Series | None = None) -> dict:
    params = {
        "subsample": 0.85,
        "colsample_bytree": 0.58,
        "max_depth": 4,
        "min_child_weight": 1.5,
        "learning_rate": 0.018,
        "max_bin": 1024,
        "tree_method": "hist",
        "device": "cuda",
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "random_state": seed,
        "n_estimators": 18000,
        "early_stopping_rounds": 450,
        "reg_alpha": 0.02,
        "reg_lambda": 1.6,
    }

    # Use AUC-PR only for the minority OVR classifiers as a mild objective-side variation.
    if target_id in [1, 2]:
        params["eval_metric"] = "aucpr"

    # Mild class imbalance correction. Use a power < 1 to avoid destroying calibration.
    if y_train is not None:
        pos = float(np.sum(y_train))
        neg = float(len(y_train) - pos)
        if pos > 0:
            params["scale_pos_weight"] = float((neg / pos) ** SPW_POWER)

    return params

def fit_binary_xgb_for_class(target_id: int, seed: int):
    target_name = reverse_mapping[target_id]
    skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=seed + target_id * 101)

    oof_class = np.zeros(len(features), dtype=np.float64)
    pred_class = np.zeros(len(test_features), dtype=np.float64)
    logs = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(features, target_ovr[target_name]), start=1):
        fold_seed = int(seed + target_id * 1009 + fold * FOLD_SEED_MULTIPLIER)

        X_train = features.iloc[tr_idx].reset_index(drop=True)
        X_val = features.iloc[val_idx].reset_index(drop=True)
        X_test = test_features.copy()

        y_train = target_ovr[target_name].iloc[tr_idx].reset_index(drop=True)
        y_val = target_ovr[target_name].iloc[val_idx].reset_index(drop=True)

        # Fold-safe TargetEncoder for combo features.
        if len(TE_columns) > 0:
            encoder = TargetEncoder(cv=5, random_state=fold_seed, smooth="auto")

            te_train = pd.DataFrame(
                encoder.fit_transform(X_train[TE_columns], y_train),
                columns=[f"TE_{c}" for c in TE_columns],
            )
            te_val = pd.DataFrame(
                encoder.transform(X_val[TE_columns]),
                columns=[f"TE_{c}" for c in TE_columns],
            )
            te_test = pd.DataFrame(
                encoder.transform(X_test[TE_columns]),
                columns=[f"TE_{c}" for c in TE_columns],
            )

            X_train = pd.concat([te_train.reset_index(drop=True), X_train.reset_index(drop=True)], axis=1)
            X_val = pd.concat([te_val.reset_index(drop=True), X_val.reset_index(drop=True)], axis=1)
            X_test = pd.concat([te_test.reset_index(drop=True), X_test.reset_index(drop=True)], axis=1)

            X_train = X_train.drop(TE_columns, axis=1)
            X_val = X_val.drop(TE_columns, axis=1)
            X_test = X_test.drop(TE_columns, axis=1)

        X_train.columns = X_train.columns.astype(str)
        X_val.columns = X_val.columns.astype(str)
        X_test.columns = X_test.columns.astype(str)

        params = xgb_params_for(fold_seed, target_id=target_id, y_train=y_train)

        print(f"\n[target={target_name}] seed={seed} fold={fold} fold_seed={fold_seed}")
        print("eval_metric:", params.get("eval_metric"), "scale_pos_weight:", params.get("scale_pos_weight"))
        model = XGBClassifier(**params)
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=1000,
        )

        val_pred = model.predict_proba(X_val)[:, 1]
        test_pred = model.predict_proba(X_test)[:, 1]

        oof_class[val_idx] = val_pred
        pred_class += test_pred / FOLDS

        auc = roc_auc_score(y_val, val_pred)
        best_iteration = getattr(model, "best_iteration", None)
        logs.append({
            "target_id": target_id,
            "target_name": target_name,
            "seed": int(seed),
            "fold_seed": int(fold_seed),
            "fold": int(fold),
            "auc": float(auc),
            "eval_metric": params.get("eval_metric"),
            "scale_pos_weight": float(params.get("scale_pos_weight", np.nan)),
            "best_iteration": None if best_iteration is None else int(best_iteration),
            "n_train": int(len(tr_idx)),
            "n_valid": int(len(val_idx)),
            "n_features_after_te": int(X_train.shape[1]),
        })
        print(f"AUC: {auc:.6f}, best_iteration={best_iteration}")

        del model, X_train, X_val, X_test, y_train, y_val
        gc.collect()

    return oof_class, pred_class, logs

all_oof_by_seed = []
all_pred_by_seed = []
model_logs = []

for seed in SEED_LIST:
    print("=" * 80)
    print("SEED:", seed)
    oof_seed = np.zeros((len(features), 3), dtype=np.float64)
    pred_seed = np.zeros((len(test_features), 3), dtype=np.float64)

    for target_id in [0, 1, 2]:
        oof_class, pred_class, logs = fit_binary_xgb_for_class(target_id=target_id, seed=seed)
        oof_seed[:, target_id] = oof_class
        pred_seed[:, target_id] = pred_class
        model_logs.extend(logs)

    # Normalize independent one-vs-rest probabilities into multiclass probabilities.
    oof_seed = oof_seed / np.clip(oof_seed.sum(axis=1, keepdims=True), 1e-12, None)
    pred_seed = pred_seed / np.clip(pred_seed.sum(axis=1, keepdims=True), 1e-12, None)

    seed_cv = balanced_accuracy_score(y, np.argmax(oof_seed, axis=1))
    print(f"Seed {seed} normalized OOF Balanced Accuracy: {seed_cv:.9f}")

    all_oof_by_seed.append(oof_seed)
    all_pred_by_seed.append(pred_seed)

oof_proba = np.mean(all_oof_by_seed, axis=0)
pred_proba = np.mean(all_pred_by_seed, axis=0)

# Normalize again after seed averaging.
oof_proba = oof_proba / np.clip(oof_proba.sum(axis=1, keepdims=True), 1e-12, None)
pred_proba = pred_proba / np.clip(pred_proba.sum(axis=1, keepdims=True), 1e-12, None)

model_log_df = pd.DataFrame(model_logs)
model_log_df.to_csv(MODEL_LOG_CSV, index=False)

overall_cv = balanced_accuracy_score(y, np.argmax(oof_proba, axis=1))
print("\nOverall normalized OOF Balanced Accuracy:", overall_cv)
display(model_log_df.head())
display(model_log_df.groupby(["target_name", "seed"])["auc"].mean().reset_index())


SEED: 2027

[target=GALAXY] seed=2027 fold=1 fold_seed=2044
eval_metric: auc scale_pos_weight: 0.8530242536178201
[0]	validation_0-auc:0.97828
[1000]	validation_0-auc:0.99497
[2000]	validation_0-auc:0.99542
[3000]	validation_0-auc:0.99556
[4000]	validation_0-auc:0.99562
[5000]	validation_0-auc:0.99566
[6000]	validation_0-auc:0.99569
[7000]	validation_0-auc:0.99570
[8000]	validation_0-auc:0.99570
[8451]	validation_0-auc:0.99570


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [05:36:45] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


AUC: 0.995702, best_iteration=8001

[target=GALAXY] seed=2027 fold=2 fold_seed=2061
eval_metric: auc scale_pos_weight: 0.8530242536178201
[0]	validation_0-auc:0.97484
[1000]	validation_0-auc:0.99473
[2000]	validation_0-auc:0.99515
[3000]	validation_0-auc:0.99529
[4000]	validation_0-auc:0.99536
[5000]	validation_0-auc:0.99539
[6000]	validation_0-auc:0.99541
[7000]	validation_0-auc:0.99542
[8000]	validation_0-auc:0.99543
[8378]	validation_0-auc:0.99542
AUC: 0.995428, best_iteration=7928

[target=GALAXY] seed=2027 fold=3 fold_seed=2078
eval_metric: auc scale_pos_weight: 0.8530255873570285
[0]	validation_0-auc:0.97530
[1000]	validation_0-auc:0.99518
[2000]	validation_0-auc:0.99553
[3000]	validation_0-auc:0.99565
[4000]	validation_0-auc:0.99571
[5000]	validation_0-auc:0.99575
[6000]	validation_0-auc:0.99577
[7000]	validation_0-auc:0.99578
[7206]	validation_0-auc:0.99578
AUC: 0.995783, best_iteration=6756

[target=GALAXY] seed=2027 fold=4 fold_seed=2095
eval_metric: auc scale_pos_weight: 0.8

,target_id,target_name,seed,fold_seed,fold,auc,eval_metric,scale_pos_weight,best_iteration,n_train,n_valid,n_features_after_te
0,0,GALAXY,2027,2044,1,0.995702,auc,0.853024,8001,461877,115470,40
1,0,GALAXY,2027,2061,2,0.995428,auc,0.853024,7928,461877,115470,40
2,0,GALAXY,2027,2078,3,0.995783,auc,0.853026,6756,461878,115469,40
3,0,GALAXY,2027,2095,4,0.995616,auc,0.853026,8137,461878,115469,40
4,0,GALAXY,2027,2112,5,0.995609,auc,0.853026,8476,461878,115469,40


,target_name,seed,auc
0,GALAXY,2027,0.995628
1,GALAXY,3407,0.995624
2,GALAXY,7117,0.995639
3,QSO,2027,0.997925
4,QSO,3407,0.997911
5,QSO,7117,0.997908
6,STAR,2027,0.996825
7,STAR,3407,0.996855
8,STAR,7117,0.996837


In [7]:
# ============================================================
# 6. CV diagnostics / save outputs
# ============================================================
oof_pred_label = np.argmax(oof_proba, axis=1)
recalls = recall_score(y, oof_pred_label, average=None)
cm = confusion_matrix(y, oof_pred_label)

eval_skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
fold_records = []
for fold, (_, val_idx) in enumerate(eval_skf.split(oof_proba, y), start=1):
    fold_score = balanced_accuracy_score(y[val_idx], oof_pred_label[val_idx])
    fold_recalls = recall_score(y[val_idx], oof_pred_label[val_idx], average=None)
    fold_records.append({
        "fold": fold,
        "balanced_accuracy": float(fold_score),
        "recall_GALAXY": float(fold_recalls[0]),
        "recall_QSO": float(fold_recalls[1]),
        "recall_STAR": float(fold_recalls[2]),
        "n_valid": int(len(val_idx)),
    })

fold_scores = pd.DataFrame(fold_records)
fold_std = float(fold_scores["balanced_accuracy"].std(ddof=0))
fold_scores.to_csv(FOLD_SCORES_CSV, index=False)

np.save(OOF_NPY, oof_proba)
np.save(PRED_NPY, pred_proba)

oof_df = pd.DataFrame(oof_proba, columns=CLASS_NAMES)
oof_df.insert(0, ID, train_full["id"].values)
oof_df[TARGET] = [reverse_mapping[v] for v in y]
oof_df.to_csv(OOF_CSV, index=False)

pred_df = pd.DataFrame(pred_proba, columns=CLASS_NAMES)
pred_df.insert(0, ID, test_ids)
pred_df.to_csv(PRED_CSV, index=False)

test_pred_label = np.argmax(pred_proba, axis=1)
test_labels = [reverse_mapping[p] for p in test_pred_label]
submission = pd.DataFrame({ID: test_ids, TARGET: test_labels})
submission.to_csv(SUB_CSV, index=False)

submission_counts = submission[TARGET].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int)
submission_ratios = (submission_counts / len(submission)).round(8)
pd.DataFrame({
    "class": CLASS_NAMES,
    "count": [int(submission_counts[c]) for c in CLASS_NAMES],
    "ratio": [float(submission_ratios[c]) for c in CLASS_NAMES],
}).to_csv(CLASS_DIST_CSV, index=False)

cv_result = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "created_at": "2026-06-14",
    "status": "completed",
    "metric": "balanced_accuracy_score",
    "model_family": "XGBoost",
    "model_type": "ovr_xgb_seed2027_fe_foldvariant",
    "model": "XGBClassifier binary one-vs-rest",
    "source": "exp_20260608_030_ovr_xgb_as_is",
    "purpose": "Return to the 030 OVR route after 047 multiclass failed; create an XGB-family material with conservative FE, seed/fold changes, and mild class weighting.",
    "seed": SEED,
    "seed_list": SEED_LIST,
    "n_splits": FOLDS,
    "fold_seed_multiplier": FOLD_SEED_MULTIPLIER,
    "spw_power": SPW_POWER,
    "overall_cv": float(overall_cv),
    "fold_std": fold_std,
    "recalls": {
        "GALAXY": float(recalls[0]),
        "QSO": float(recalls[1]),
        "STAR": float(recalls[2]),
    },
    "confusion_matrix": cm.tolist(),
    "variant_design": {
        "base_exp_id": "exp_20260608_030_ovr_xgb_as_is",
        "failed_previous_xgb_variant": "exp_20260614_047_xgb_multiclass_fe_objective_foldvariant",
        "changed_from_030": [
            "kept objective: binary one-vs-rest",
            "seed/folds: SEED=2027 and seed_list=[2027,3407,7117]",
            "fold_seed_formula: seed + target_id*1009 + fold*17",
            "feature engineering: add adjacent colors, mag stats, redshift transforms, and one redshift quantile bin",
            "target encoding: add one extra redshift/g combo and use fold_seed as random_state",
            "minority classes: use aucpr eval_metric for QSO/STAR",
            "mild class imbalance correction: scale_pos_weight=(neg/pos)^0.25",
        ],
    },
    "xgb_params_base": xgb_params_for(SEED_LIST[0], target_id=2, y_train=None),
    "feature_info": feature_info,
    "fold_scores": fold_scores.to_dict(orient="records"),
    "binary_model_log": model_log_df.to_dict(orient="records"),
    "submission_distribution": {
        "total_rows": int(len(submission)),
        "class_counts": {k: int(v) for k, v in submission_counts.to_dict().items()},
        "class_ratio": {k: float(v) for k, v in submission_ratios.to_dict().items()},
    },
    "outputs": {
        "oof_proba": OOF_NPY.name,
        "pred_proba": PRED_NPY.name,
        "oof_csv": OOF_CSV.name,
        "pred_csv": PRED_CSV.name,
        "submission": SUB_CSV.name,
        "cv_result": CV_RESULT_JSON.name,
        "fold_scores": FOLD_SCORES_CSV.name,
        "feature_info": FEATURE_INFO_JSON.name,
        "feature_columns": FEATURE_COLUMNS_CSV.name,
        "binary_model_log": MODEL_LOG_CSV.name,
        "class_distribution": CLASS_DIST_CSV.name,
    },
}

with open(CV_RESULT_JSON, "w", encoding="utf-8") as f:
    json.dump(cv_result, f, indent=2, ensure_ascii=False, default=str)

print("overall_cv:", overall_cv)
print("fold_std:", fold_std)
print("recalls:", dict(zip(CLASS_NAMES, recalls)))
print("saved:", OOF_NPY.name, PRED_NPY.name, SUB_CSV.name)
display(fold_scores)
display(submission.head())


overall_cv: 0.9636973807714805
fold_std: 0.0007384915448554254
recalls: {'GALAXY': np.float64(0.9769153332626894), 'QSO': np.float64(0.9681756485662822), 'STAR': np.float64(0.9460011604854698)}
saved: oof_exp_20260614_048_ovr_xgb_seed2027_fe_foldvariant_proba.npy pred_exp_20260614_048_ovr_xgb_seed2027_fe_foldvariant_proba.npy submission_exp_20260614_048_ovr_xgb_seed2027_fe_foldvariant.csv


,fold,balanced_accuracy,recall_GALAXY,recall_QSO,recall_STAR,n_valid
0,1,0.963023,0.977509,0.967348,0.944213,115470
1,2,0.964068,0.976807,0.968586,0.946812,115470
2,3,0.964714,0.977257,0.969653,0.947232,115469
3,4,0.963992,0.976568,0.968414,0.946993,115469
4,5,0.962690,0.976436,0.966877,0.944757,115469


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


In [8]:
# ============================================================
# 7. Update registry / memo
# ============================================================
registry_path = WORKDIR / "oof_registry.csv"

registry_row = {
    "exp_id": EXP_ID,
    "model_family": "XGBoost",
    "feature_family": "ovr_xgb_seed2027_conservative_fe_foldvariant",
    "cv_metric": "balanced_accuracy_score",
    "cv_score": float(overall_cv),
    "fold_std": float(fold_scores["balanced_accuracy"].std(ddof=0)),
    "public_lb": np.nan,
    "use_original": False,
    "use_fe": True,
    "use_bias_search": False,
    "oof_path": str(OOF_NPY),
    "pred_path": str(PRED_NPY),
    "submission_path": str(SUB_CSV),
    "role": "xgb_ovr_family_material_after_failed_multiclass",
    "status": "completed",
    "keep_hold_drop": "pending_public_lb_and_corr",
    "notes": (
        "048 returns to the 030 OVR binary XGB route after 047 multiclass softprob was weak. "
        "It keeps binary:logistic one-vs-rest, changes seed/fold, adds conservative FE, "
        "uses fold-specific TargetEncoder seeds, uses aucpr for QSO/STAR, and applies mild scale_pos_weight."
    ),
}

if registry_path.exists():
    registry = pd.read_csv(registry_path)
    registry = registry[registry["exp_id"] != EXP_ID]
    registry = pd.concat([registry, pd.DataFrame([registry_row])], ignore_index=True)
else:
    registry = pd.DataFrame([registry_row])

registry.to_csv(registry_path, index=False)
registry.to_csv(OUTDIR / "oof_registry.csv", index=False)

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "OVR XGB seed2027 conservative FE/fold variant",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "model": "XGBClassifier binary one-vs-rest",
        "created_at": "2026-06-14",
    },
    "objective": {
        "summary": (
            "After 047 multiclass softprob XGB underperformed, return to the 030 binary OVR XGB route. "
            "Create a conservative XGB-family material by changing seed/folds and modestly expanding FE while preserving the OVR objective."
        ),
        "success_criteria": [
            "keep one-vs-rest binary XGB for GALAXY/QSO/STAR",
            "do not use multiclass softprob",
            "use seed_list [2027,3407,7117]",
            "use conservative FE changes from 030",
            "use fold-safe TargetEncoder",
            "save OOF proba npy with exp_id in filename",
            "save test pred proba npy with exp_id in filename",
            "save submission",
            "save cv_result",
            "save fold_scores",
            "update oof_registry",
            "write memo.yml",
        ],
    },
    "data": {
        "train_path": str(TRAIN_PATH),
        "test_path": str(TEST_PATH),
        "sample_submission_path": str(SAMPLE_SUB_PATH),
        "target_col": TARGET,
        "id_col": ID,
        "use_original": False,
    },
    "seed": {
        "base_seed": SEED,
        "seed_list": SEED_LIST,
        "numpy_seed": SEED,
        "fold_seed_formula": "seed + target_id*1009 + fold*17",
        "stratified_kfold_random_state": "seed + target_id*101 for each binary classifier",
        "target_encoder_random_state": "fold_seed",
    },
    "features": {
        "feature_family": "ovr_xgb_seed2027_conservative_fe_foldvariant",
        "source_exp": "exp_20260608_030_ovr_xgb_as_is",
        "feature_info": feature_info,
        "changed_from_030": [
            "add g-r/r-i/i-z/g-i/r-z color indices",
            "add magnitude mean/std/range and u-to-z slope",
            "add redshift_abs and log1p_redshift",
            "add z/g ratio",
            "keep delta 100/500 qbins",
            "add redshift 64 qbin",
            "add redshift/g combo for TE",
        ],
        "target_encoder_safety": (
            "TargetEncoder is fit inside each binary fold on train fold combo features and binary labels, "
            "then transforms validation and test."
        ),
    },
    "model": {
        "family": "XGBoost",
        "type": "binary one-vs-rest",
        "class_order": CLASS_NAMES,
        "xgb_params_base": xgb_params_for(SEED_LIST[0], target_id=2, y_train=None),
        "n_binary_models": int(len(SEED_LIST) * len(CLASS_NAMES) * FOLDS),
        "probability_postprocess": (
            "Average across seeds for each class, then normalize independent one-vs-rest probabilities "
            "so each row sums to 1."
        ),
        "target_specific_changes": [
            "QSO/STAR use aucpr as eval_metric",
            "scale_pos_weight=(neg/pos)^0.25 per binary fold",
        ],
        "excluded": [
            "external stacking files",
            "LogisticRegressionCV second-stage blend",
            "final bias search as submitted output",
            "multiclass softprob objective",
        ],
    },
    "cv": {
        "scheme": "StratifiedKFold binary one-vs-rest per class and seed",
        "n_splits": FOLDS,
        "shuffle": True,
        "score": float(overall_cv),
        "fold_scores": fold_scores.to_dict(orient="records"),
        "fold_std": float(fold_scores["balanced_accuracy"].std(ddof=0)),
        "recalls": {
            "GALAXY": float(recalls[0]),
            "QSO": float(recalls[1]),
            "STAR": float(recalls[2]),
        },
        "binary_model_log_path": MODEL_LOG_CSV.name,
    },
    "submission_distribution": {
        "total_rows": int(len(submission)),
        "class_counts": {k: int(v) for k, v in submission_counts.to_dict().items()},
        "class_ratio": {k: float(v) for k, v in submission_ratios.to_dict().items()},
    },
    "outputs": {
        "oof_proba": OOF_NPY.name,
        "pred_proba": PRED_NPY.name,
        "oof_csv": OOF_CSV.name,
        "pred_csv": PRED_CSV.name,
        "submission": SUB_CSV.name,
        "cv_result": CV_RESULT_JSON.name,
        "fold_scores": FOLD_SCORES_CSV.name,
        "feature_info": FEATURE_INFO_JSON.name,
        "feature_columns": FEATURE_COLUMNS_CSV.name,
        "binary_model_log": MODEL_LOG_CSV.name,
        "class_distribution": CLASS_DIST_CSV.name,
        "registry": "oof_registry.csv",
    },
    "judgement": {
        "status": "pending_public_lb_and_corr",
        "reason": (
            "048 is a conservative OVR XGB variant. It should be compared first against 030 and 047. "
            "Only if CV/LB or correlation behavior is acceptable should it be added to GPU LogReg stacker."
        ),
        "next_action": [
            f"Submit {SUB_CSV.name}",
            "Record Public LB",
            "Add OOF/pred npy to npy-files dataset only if worth testing further",
            "Compare against 030 and 047",
            "If favorable, run 049 GPU LogReg stacker add048",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("registry saved:", registry_path)
print("memo saved:", memo_path)
display(registry.tail())


registry saved: /kaggle/working/oof_registry.csv
memo saved: /kaggle/working/exp_20260614_048_ovr_xgb_seed2027_fe_foldvariant/memo.yml


,exp_id,model_family,feature_family,cv_metric,cv_score,fold_std,public_lb,use_original,use_fe,use_bias_search,oof_path,pred_path,submission_path,role,status,keep_hold_drop,notes
0,exp_20260614_048_ovr_xgb_seed2027_fe_foldvariant,XGBoost,ovr_xgb_seed2027_conservative_fe_foldvariant,balanced_accuracy_score,0.963697,0.000738,NaN,False,True,False,/kaggle/working/exp_20260614_048_ovr_xgb_seed2...,/kaggle/working/exp_20260614_048_ovr_xgb_seed2...,/kaggle/working/exp_20260614_048_ovr_xgb_seed2...,xgb_ovr_family_material_after_failed_multiclass,completed,pending_public_lb_and_corr,048 returns to the 030 OVR binary XGB route af...
